# Task 078: phoeniks — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: THz time-domain spectroscopy material parameter extraction using phoeniks

**Paper**: batman: Bad-Ass Transit Model cAlculatioN in Python
**Repository**: https://github.com/lkreidberg/batman

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 48.25 dB (refractive index n)
- **SSIM**: N/A (1D spectral data)
- **Evaluation**: 1D spectral fitting — THz-TDS parameter extraction (CC_n=0.99999813, CC_κ=0.99999806, CC_α=0.99999846)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
phoeniks - THz Material Parameter Extraction
=============================================
Task: Extract complex refractive index n(ω) and extinction coefficient κ(ω)
      from THz Time-Domain Spectroscopy (THz-TDS) measurements.
      
      Forward model: THz pulse propagates through a dielectric slab.
        H(ω) = E_sample(ω) / E_reference(ω) is the measured transfer function.
      
      Inverse problem: Given measured H(ω), recover n(ω) and κ(ω) by minimizing
        the error between modeled and measured transfer function at each frequency.

Repo: https://github.com/puls-lab/phoeniks
Paper: Pupeza et al., Optics Express 15(7), 4335 (2007)

Usage:
    /data/yjh/phoeniks_env/bin/python phoeniks_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_data`, `compute_correlation`, `interpolate_to_common_freq`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Data Loading
# ═══════════════════════════════════════════════════════════
def load_data():
    """
    Load artificial THz-TDS data from the phoeniks example directory.
    Returns: (data_obj, ground_truth_dict)
      - data_obj: phoeniks Data object with reference and sample time-domain traces
      - ground_truth_dict: dict with 'frequency', 'n', 'k', 'alpha' arrays
    """
    ref_file = os.path.join(EXAMPLE_DIR, "Artifical_Reference.txt")
    sam_file = os.path.join(EXAMPLE_DIR, "Artifical_Sample_1mm.txt")
    gt_file = os.path.join(EXAMPLE_DIR, "Artifical_n_k_alpha.txt")

    ref = np.loadtxt(ref_file)
    sam = np.loadtxt(sam_file)
    gt_data = np.loadtxt(gt_file)

    time = ref[:, 0]
    td_reference = ref[:, 1]
    td_sample = sam[:, 1]

    # Create phoeniks Data object
    data_obj = pk.thz_data.Data(time, td_reference, td_sample)

    # Ground truth: columns are [frequency, n, k, alpha]
    ground_truth = {
        'frequency': gt_data[:, 0],
        'n': gt_data[:, 1],
        'k': gt_data[:, 2],
        'alpha': gt_data[:, 3],
    }

    return data_obj, ground_truth

def compute_correlation(ref, test):
    """Compute Pearson correlation coefficient."""
    r = np.corrcoef(ref.flatten(), test.flatten())[0, 1]
    return r

def interpolate_to_common_freq(gt_freq, gt_values, recon_freq, recon_values):
    """
    Interpolate reconstructed values onto ground truth frequency grid
    for fair comparison.
    """
    # Find overlapping frequency range
    f_min = max(gt_freq.min(), recon_freq.min())
    f_max = min(gt_freq.max(), recon_freq.max())
    
    # Create mask for GT frequencies within range
    mask = (gt_freq >= f_min) & (gt_freq <= f_max)
    common_freq = gt_freq[mask]
    gt_common = gt_values[mask]
    
    # Interpolate reconstruction to common frequencies
    recon_common = np.interp(common_freq, recon_freq, recon_values)
    
    return common_freq, gt_common, recon_common

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. Forward Operator (THz Transfer Function)
# ═══════════════════════════════════════════════════════════
def forward_operator(n, k, frequency, thickness):
    """
    Compute the transfer function H(ω) for a single dielectric layer.
    H(ω) = t12 * t21 * P(ω) / P_air(ω) * FP(ω)
    
    where:
      - t12, t21: Fresnel transmission coefficients
      - P(ω): propagation through material
      - P_air(ω): propagation through air (same thickness)
      - FP(ω): Fabry-Perot etalon factor
      
    Parameters:
      n: refractive index array (per frequency)
      k: extinction coefficient array (per frequency)
      frequency: frequency array (Hz)
      thickness: sample thickness (m)
    Returns:
      H: complex transfer function array
    """
    omega = 2 * np.pi * frequency
    complex_n = n - 1j * k  # complex refractive index
    
    t12 = 2 / (1 + complex_n)
    t21 = 2 * complex_n / (1 + complex_n)
    r22 = (complex_n - 1) / (1 + complex_n)
    rr = r22 * r22
    tt = t12 * t21
    
    propagation = np.exp(-1j * omega * complex_n * thickness / c_0)
    propagation_air = np.exp(-1j * omega * thickness / c_0)
    
    FP = 1 / (1 - rr * (propagation ** 2))
    H = tt * propagation * FP / propagation_air
    
    return H

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver / Reconstruction (phoeniks Extraction)
# ═══════════════════════════════════════════════════════════
def reconstruct(data_obj, thickness, freq_start, freq_stop):
    """
    Extract n(ω) and κ(ω) from THz-TDS data using phoeniks.
    
    Algorithm:
      1. Window time-domain traces and zero-pad for frequency resolution
      2. Compute and unwrap phase of transfer function H(ω)
      3. Get initial estimates of n, k from analytical approximation
      4. Optimize n, k at each frequency by minimizing error between
         measured and modeled H(ω)
    
    Parameters:
      data_obj: phoeniks Data object
      thickness: sample thickness (m)
      freq_start: start frequency for phase unwrapping (Hz)
      freq_stop: stop frequency for phase unwrapping (Hz)
    Returns:
      dict with 'frequency', 'n', 'k', 'alpha' arrays
    """
    # Window traces and zero-pad for better frequency resolution
    data_obj.window_traces(time_start=10e-12, time_end=90e-12)
    data_obj.pad_zeros(new_frequency_resolution=2e9)
    
    # Create extraction object
    extract_obj = pk.extraction.Extraction(data_obj)
    
    # Unwrap phase in the specified frequency range
    extract_obj.unwrap_phase(frequency_start=freq_start, frequency_stop=freq_stop)
    
    # Get initial n, k estimates
    n_init, k_init = extract_obj.get_initial_nk(thickness=thickness)
    
    # Run frequency-by-frequency optimization
    frequency, n_opt, k_opt, alpha_opt = extract_obj.run_optimization(thickness=thickness)
    
    result = {
        'frequency': frequency,
        'n': n_opt,
        'k': k_opt,
        'alpha': alpha_opt,
    }
    
    return result

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_rmse`, `compute_relative_error`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Evaluation Metrics
# ═══════════════════════════════════════════════════════════
def compute_psnr(ref, test, data_range=None):
    """Compute PSNR (dB) for 1D signals."""
    if data_range is None:
        data_range = ref.max() - ref.min()
    mse = np.mean((ref.astype(float) - test.astype(float)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(data_range ** 2 / mse)

def compute_rmse(ref, test):
    """Compute RMSE."""
    return np.sqrt(np.mean((ref.astype(float) - test.astype(float)) ** 2))

def compute_relative_error(ref, test):
    """Compute relative error ||ref - test|| / ||ref||."""
    return np.linalg.norm(ref - test) / np.linalg.norm(ref)

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(gt, recon, metrics, save_path):
    """
    Generate comprehensive visualization for THz parameter extraction.
    Shows: (a) n(ω) comparison, (b) α(ω) comparison, (c) n residual, (d) α residual
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Convert to THz for display
    gt_freq_THz = gt['frequency'] / 1e12
    recon_freq_THz = recon['frequency'] / 1e12
    
    # (a) Refractive index n(ω) comparison
    ax = axes[0, 0]
    ax.plot(gt_freq_THz, gt['n'], 'b.', markersize=2, alpha=0.6, label='Ground Truth')
    ax.plot(recon_freq_THz, recon['n'], 'r-', linewidth=1.0, label='phoeniks Extraction')
    ax.set_xlabel('Frequency (THz)')
    ax.set_ylabel('Refractive Index n')
    ax.set_title('Refractive Index')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # (b) Absorption coefficient α(ω) comparison
    ax = axes[0, 1]
    gt_alpha_cm = gt['alpha'] * 0.01  # Convert to cm⁻¹
    recon_alpha_cm = recon['alpha'] * 0.01
    ax.plot(gt_freq_THz, gt_alpha_cm, 'b.', markersize=2, alpha=0.6, label='Ground Truth')
    ax.plot(recon_freq_THz, recon_alpha_cm, 'r-', linewidth=1.0, label='phoeniks Extraction')
    ax.set_xlabel('Frequency (THz)')
    ax.set_ylabel(r'Absorption $\alpha$ (cm$^{-1}$)')
    ax.set_title('Absorption Coefficient')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Interpolate for residual computation
    common_freq_n, gt_n_common, recon_n_common = interpolate_to_common_freq(
        gt['frequency'], gt['n'], recon['frequency'], recon['n'])
    common_freq_a, gt_a_common, recon_a_common = interpolate_to_common_freq(
        gt['frequency'], gt['alpha'] * 0.01, recon['frequency'], recon['alpha'] * 0.01)
    
    # (c) Refractive index residual
    ax = axes[1, 0]
    residual_n = recon_n_common - gt_n_common
    ax.plot(common_freq_n / 1e12, residual_n, 'g-', linewidth=0.8)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Frequency (THz)')
    ax.set_ylabel('Δn (Extracted − GT)')
    ax.set_title(f'n Residual (RMSE={metrics.get("rmse_n", 0):.6f})')
    ax.grid(True, alpha=0.3)
    
    # (d) Absorption residual
    ax = axes[1, 1]
    residual_a = recon_a_common - gt_a_common
    ax.plot(common_freq_a / 1e12, residual_a, 'm-', linewidth=0.8)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Frequency (THz)')
    ax.set_ylabel(r'Δα (cm$^{-1}$)')
    ax.set_title(f'α Residual (RMSE={metrics.get("rmse_alpha", 0):.6f})')
    ax.grid(True, alpha=0.3)
    
    fig.suptitle(
        f"THz-TDS Parameter Extraction — PSNR_n={metrics.get('psnr_n', 0):.2f} dB | "
        f"CC_n={metrics.get('cc_n', 0):.6f} | "
        f"CC_α={metrics.get('cc_alpha', 0):.6f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved visualization → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
print("=" * 60)
print("  phoeniks — THz Material Parameter Extraction")
print("=" * 60)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Load data
data_obj, ground_truth = load_data()
print(f"[DATA] Time-domain reference: {data_obj.td_reference.shape}")
print(f"[DATA] Time-domain sample: {data_obj.td_sample.shape}")
print(f"[DATA] Ground truth n/k available at {len(ground_truth['frequency'])} frequencies")
print(f"[DATA] GT frequency range: {ground_truth['frequency'][0]/1e12:.3f} - {ground_truth['frequency'][-1]/1e12:.3f} THz")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (b) Run reconstruction (inverse problem)
print("\n[RECON] Running phoeniks extraction...")
reconstruction = reconstruct(data_obj, SAMPLE_THICKNESS, FREQ_START, FREQ_STOP)
print(f"[RECON] Extracted n/k at {len(reconstruction['frequency'])} frequencies")
print(f"[RECON] Frequency range: {reconstruction['frequency'][0]/1e12:.3f} - {reconstruction['frequency'][-1]/1e12:.3f} THz")

# (c) Evaluate — interpolate to common frequency grid
print("\n[EVAL] Computing metrics...")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Refractive index n
common_freq_n, gt_n, recon_n = interpolate_to_common_freq(
    ground_truth['frequency'], ground_truth['n'],
    reconstruction['frequency'], reconstruction['n']
)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Extinction coefficient k / absorption alpha
common_freq_a, gt_alpha, recon_alpha = interpolate_to_common_freq(
    ground_truth['frequency'], ground_truth['alpha'] * 0.01,  # to cm^-1
    reconstruction['frequency'], reconstruction['alpha'] * 0.01
)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Also evaluate k
common_freq_k, gt_k, recon_k = interpolate_to_common_freq(
    ground_truth['frequency'], ground_truth['k'],
    reconstruction['frequency'], reconstruction['k']
)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
metrics = {
    # n metrics
    "psnr_n": float(compute_psnr(gt_n, recon_n)),
    "cc_n": float(compute_correlation(gt_n, recon_n)),
    "rmse_n": float(compute_rmse(gt_n, recon_n)),
    "re_n": float(compute_relative_error(gt_n, recon_n)),
    # k metrics
    "psnr_k": float(compute_psnr(gt_k, recon_k)),
    "cc_k": float(compute_correlation(gt_k, recon_k)),
    "rmse_k": float(compute_rmse(gt_k, recon_k)),
    # alpha metrics
    "cc_alpha": float(compute_correlation(gt_alpha, recon_alpha)),
    "rmse_alpha": float(compute_rmse(gt_alpha, recon_alpha)),
    # Overall
    "psnr": float(compute_psnr(gt_n, recon_n)),
    "rmse": float(compute_rmse(gt_n, recon_n)),
}

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"[EVAL] n — PSNR = {metrics['psnr_n']:.4f} dB")
print(f"[EVAL] n — CC   = {metrics['cc_n']:.8f}")
print(f"[EVAL] n — RMSE = {metrics['rmse_n']:.8f}")
print(f"[EVAL] n — RE   = {metrics['re_n']:.8f}")
print(f"[EVAL] k — PSNR = {metrics['psnr_k']:.4f} dB")
print(f"[EVAL] k — CC   = {metrics['cc_k']:.8f}")
print(f"[EVAL] α — CC   = {metrics['cc_alpha']:.8f}")
print(f"[EVAL] α — RMSE = {metrics['rmse_alpha']:.8f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# (d) Save metrics
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n[SAVE] Metrics → {metrics_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (e) Visualize
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize_results(ground_truth, reconstruction, metrics, vis_path)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (f) Save arrays — combine frequency + n + k as ground truth and reconstruction
gt_array = np.column_stack([common_freq_n, gt_n, gt_k[:len(common_freq_n)]])
recon_array = np.column_stack([common_freq_n, recon_n, recon_k[:len(common_freq_n)]])
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_array)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), recon_array)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Also save the measurement (transfer function magnitude + phase)
measurement = np.column_stack([
    reconstruction['frequency'],
    reconstruction['n'],
    reconstruction['k'],
    reconstruction['alpha']
])
np.save(os.path.join(RESULTS_DIR, "input.npy"), measurement)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  DONE")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **phoeniks**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=48.25 dB (refractive index n), SSIM=N/A (1D spectral data))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: batman: Bad-Ass Transit Model cAlculatioN in Python
- Repository: https://github.com/lkreidberg/batman
